In [ ]:
# 클로드와 함께 메모리 성능을 고려한 소스 2025.12.04

32GB RAM 환경에 최적화된 **HashingVectorizer 버전** 

## 🎯 핵심 개선사항

### 1. **HashingVectorizer의 메모리 효율성**
```python
# 기존 TF-IDF (메모리 폭발)
vec = TfidfVectorizer(max_features=30000)
vec.fit(combined_text)  # vocabulary 저장 (수백MB)
train_vec = vec.transform(train_text)

# 개선 Hashing (메모리 고정)
hasher = HashingVectorizer(n_features=2**18)  # 262K 해시 버킷
train_vec = hasher.transform(train_text)  # fit 불필요, vocabulary 저장 안 함!
```

**메모리 절감 원리:**
- **TF-IDF**: vocabulary 딕셔너리 저장 → 메모리 가변적
- **Hashing**: 단어를 해시 함수로 고정된 버킷에 매핑 → 메모리 고정
- 32GB 환경에서 `2^18 (262,144)` 피처가 최적

### 2. **예상 실행 시간 & 메모리**
| 단계 | 시간 | 피크 메모리 |
|------|------|------------|
| 데이터 로딩 (25% 샘플) | 2-3분 | 3GB |
| Hashing 벡터화 + SVD | 10-15분 | 8GB |
| PyCaret setup | 1-2분 | 10GB |
| compare_models (상위3개) | 40-60분 | 14GB |
| blend_models | 15-20분 | 16GB |
| **총합** | **70-100분** | **최대 18GB** |

### 3. **TF-IDF vs Hashing 비교**

| 항목 | TF-IDF (기존) | HashingVectorizer |
|------|--------------|-------------------|
| 메모리 사용 | 가변적 (vocabulary 크기) | **고정적** |
| fit 필요 여부 | 필요 (시간 소요) | **불필요** |
| 피처 수 조절 | max_features | n_features (2의 거듭제곱) |
| 속도 | 느림 | **빠름** |
| 해시 충돌 | 없음 | 있음 (n_features 크면 무시 가능) |
| 피처 해석 | 가능 | 불가능 (해시값) |

### 4. **주요 파라미터 설명**

```python
# 32GB RAM 최적 설정
undersample_frac=0.25  # 25% 샘플링 (약 38만 → 9.5만 행)
n_features=2**18       # 262,144 해시 버킷 (충돌 최소화)
n_components=80        # SVD 압축 (26만 → 80차원)
fold=3                 # 3-fold CV (메모리 절약)
top_n=3                # 상위 3개 모델만 학습
```

### 5. **메모리 부족 시 추가 조정**

만약 여전히 메모리 문제가 발생하면:

```python
# 더 공격적인 설정
analyzer.load_data(undersample_frac=0.20)  # 20%로 축소

analyzer.vectorize_text_hashing(
    n_features=2**17,  # 131K로 축소
    n_components=60    # 60차원으로 축소
)

analyzer.setup_pycaret(fold=2)  # 2-fold CV

analyzer.find_and_blend_models(top_n=2)  # 상위 2개만
```

### 6. **실행 방법**

``` python
# 스크립트 실행
# python mercari_hashing_analyzer.py

# 또는 주피터 노트북에서
# from mercari_hashing_analyzer import MercariPyCaretAnalyzer
# cell 분리해서 class 선언 부분을 넣어 줌

analyzer = MercariPyCaretAnalyzer()
analyzer.load_data(undersample_frac=0.25)
analyzer.vectorize_text_hashing()
analyzer.setup_pycaret(fold=3)
analyzer.find_and_blend_models(top_n=3)
analyzer.save_metrics()
analyzer.predict_test()
```

이 버전은 **70-100분 안에 완료**되고, **최대 18GB 메모리**만 사용할 것으로 예상됩니다! 🚀

In [10]:
import pandas as pd
import numpy as np
import os
import json
import datetime
import gc
from tqdm import tqdm
import warnings
import re

warnings.filterwarnings("ignore")

from pycaret.regression import *
from sklearn.feature_extraction.text import HashingVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error


class MercariPyCaretAnalyzer:
    """
    Mercari Price Suggestion Challenge를 위한 PyCaret 기반 분석 클래스
    
    주요 개선사항:
    - TF-IDF → HashingVectorizer로 메모리 효율성 대폭 향상
    - 32GB RAM 환경에 최적화된 파라미터 설정
    - 메모리 누수 방지를 위한 적극적인 가비지 컬렉션
    """
    
    def __init__(self,
                 data_dir="../data",
                 images_dir="../images",
                 results_dir="../results"):
        self.data_dir = data_dir
        self.images_dir = images_dir
        self.results_dir = results_dir

        self.train = None
        self.test = None
        self.best_model = None
        self.setup_result = None
        self.metrics = {}

        os.makedirs(self.images_dir, exist_ok=True)
        os.makedirs(self.results_dir, exist_ok=True)

    def _collapse_rare_values(self, col, top_k, rare_label="Other"):
        """희귀값 통합 - 메모리 효율적으로 개선"""
        combined = pd.concat([self.train[col], self.test[col]], axis=0)
        value_counts = combined.value_counts()
        top_values = set(value_counts.index[:top_k])  # set으로 변환하여 검색 속도 향상
        
        del value_counts, combined
        gc.collect()

        self.train[col] = self.train[col].apply(lambda x: x if x in top_values else rare_label)
        self.test[col] = self.test[col].apply(lambda x: x if x in top_values else rare_label)

    def _simple_normalize(self, text: str) -> str:
        """텍스트 정규화"""
        text = str(text).lower()
        text = re.sub(r"[_\-\./]", " ", text)
        text = re.sub(r"\d+", " num ", text)
        text = re.sub(r"\s+", " ", text).strip()
        return text

    def _stratified_sample(self, frac=0.3, bins=10):
        """
        price 로그 변환된 데이터를 구간별로 나눠 층화 샘플링
        frac: 전체 데이터 중 몇 %를 샘플링할지
        bins: 가격 구간 개수
        """
        self.train["price_bin"] = pd.qcut(self.train["price"], q=bins, duplicates="drop")
        sampled = self.train.groupby("price_bin", group_keys=False).apply(
            lambda x: x.sample(frac=frac, random_state=23)
        )
        self.train = sampled.drop(columns=["price_bin"]).reset_index(drop=True)
        gc.collect()
        print(f"⚠️ Stratified undersampling 적용: train {self.train.shape}")

    def load_data(self, train_file="train.tsv", test_file="test.tsv", sep="\t", undersample_frac=0.25):
        """
        데이터 로딩 및 전처리
        
        undersample_frac: 32GB RAM 환경에서는 0.25(25%) 권장
        """
        print("📂 데이터 로딩 시작...")
        train_path = os.path.join(self.data_dir, train_file)
        test_path = os.path.join(self.data_dir, test_file)

        self.train = pd.read_csv(train_path, sep=sep)
        self.test = pd.read_csv(test_path, sep=sep)

        # price 로그 변환
        self.train = self.train[self.train["price"] > 0].dropna(subset=["price"])
        self.train["price"] = np.log1p(self.train["price"])

        # 층화 언더샘플링 적용
        if undersample_frac is not None:
            self._stratified_sample(frac=undersample_frac)

        # category split + 결측치 처리
        for df_name, df in [("train", self.train), ("test", self.test)]:
            # category_name 분리
            df["main_cat"], df["sub_cat"], df["sub_sub_cat"] = zip(
                *df["category_name"].apply(
                    lambda x: (x.split("/") if isinstance(x, str) and "/" in x else ["missing"]*3)
                )
            )
            
            # 결측치 처리
            df["brand_name"] = df["brand_name"].fillna("Unknown").astype(str)
            df["item_description"] = df["item_description"].fillna("No description").astype(str)
            df["name"] = df["name"].fillna("No name").astype(str)
            
            # category_name은 분리 후 불필요하므로 삭제
            df.drop(columns=["category_name"], inplace=True)

        # 희귀값 통합 (메모리 효율적으로 처리)
        print("🔄 희귀값 통합 중...")
        self._collapse_rare_values("brand_name", top_k=4000, rare_label="Other_brand")
        self._collapse_rare_values("main_cat", top_k=800, rare_label="Other_main")
        self._collapse_rare_values("sub_cat", top_k=800, rare_label="Other_sub")
        self._collapse_rare_values("sub_sub_cat", top_k=800, rare_label="Other_sub_sub")

        # 길이 피처 추가
        for df in [self.train, self.test]:
            df["name_len_char"] = df["name"].str.len()
            df["name_len_word"] = df["name"].str.split().str.len()
            df["desc_len_char"] = df["item_description"].str.len()
            df["desc_len_word"] = df["item_description"].str.split().str.len()

        # 범주형 변환
        for df in [self.train, self.test]:
            df["shipping"] = df["shipping"].astype("category")
            df["item_condition_id"] = df["item_condition_id"].astype("category")

        gc.collect()
        print(f"✅ 데이터 로드 완료: train {self.train.shape}, test {self.test.shape}")

    def vectorize_text_hashing(self, 
                               text_columns=["name", "item_description"],
                               vectorize_categorical=False,  # 범주형도 벡터화 옵션
                               n_features=2**18,
                               n_components=80):
        """
        HashingVectorizer를 사용한 텍스트 벡터화
        
        장점:
        - 메모리 사용량 고정 (vocabulary 저장 불필요)
        - fit 없이 바로 transform 가능
        - 대규모 데이터셋에 적합
        
        파라미터:
        - text_columns: 벡터화할 자유 텍스트 컬럼
        - vectorize_categorical: True시 brand_name도 텍스트로 처리
        - n_features: 2**18 (262K) - 32GB RAM 환경 최적
        - n_components: SVD 차원 축소 (80차원으로 압축)
        """
        print(f"🔍 HashingVectorizer 기반 텍스트 벡터화 시작...")
        print(f"   - n_features: {n_features:,} (메모리 고정)")
        print(f"   - SVD 압축: {n_components}차원")
        
        # 범주형도 텍스트로 처리하는 옵션
        if vectorize_categorical:
            text_columns = text_columns + ["brand_name"]
            print(f"   - brand_name도 텍스트로 벡터화")
        
        all_train_features = []
        all_test_features = []
        feature_names = []
        
        for col in tqdm(text_columns, desc="텍스트 컬럼 처리"):
            clean_col = f"{col}_clean"
            
            # 텍스트 정규화
            self.train[clean_col] = self.train[col].apply(self._simple_normalize)
            self.test[clean_col] = self.test[col].apply(self._simple_normalize)
            
            # HashingVectorizer 생성 (fit 불필요!)
            hasher = HashingVectorizer(
                n_features=n_features,
                ngram_range=(1, 2),
                norm='l2',  # L2 정규화로 스케일 통일
                alternate_sign=False,  # 해시 충돌 시 양수만 사용
                dtype=np.float32  # float64 → float32로 메모리 절약
            )
            
            # 직접 transform (fit 없이!)
            train_vec = hasher.transform(self.train[clean_col])
            test_vec = hasher.transform(self.test[clean_col])
            
            print(f"   - {col}: sparse matrix shape = {train_vec.shape}")
            
            # SVD 차원 축소 (필수! 26만 → 80차원)
            svd = TruncatedSVD(n_components=n_components, random_state=23)
            train_vec_dense = svd.fit_transform(train_vec)
            test_vec_dense = svd.transform(test_vec)
            
            explained_var = svd.explained_variance_ratio_.sum()
            print(f"   - {col}: SVD 설명력 = {explained_var:.2%}")
            
            # 희소 행렬 즉시 삭제
            del train_vec, test_vec, hasher, svd
            gc.collect()
            
            all_train_features.append(train_vec_dense)
            all_test_features.append(test_vec_dense)
            feature_names.extend([f"{col}_hash_{i}" for i in range(n_components)])
            
            # 정리된 컬럼 삭제
            self.train.drop(columns=[clean_col], inplace=True)
            self.test.drop(columns=[clean_col], inplace=True)
            gc.collect()
        
        # 최종 피처 결합
        print("🔗 피처 결합 중...")
        train_features = np.hstack(all_train_features).astype(np.float32)
        test_features = np.hstack(all_test_features).astype(np.float32)
        
        del all_train_features, all_test_features
        gc.collect()
        
        # DataFrame 생성
        self.train_vectorized = pd.DataFrame(train_features, columns=feature_names)
        self.test_vectorized = pd.DataFrame(test_features, columns=feature_names)
        
        del train_features, test_features
        gc.collect()
        
        # 추가 피처 병합
        categorical_features = [
            "main_cat", "sub_cat", "sub_sub_cat", "brand_name",
            "item_condition_id", "shipping"
        ]
        numeric_features = [
            "name_len_char", "name_len_word", "desc_len_char", "desc_len_word"
        ]
        
        # vectorize_categorical=True면 brand_name은 이미 벡터화됨
        if vectorize_categorical and "brand_name" in categorical_features:
            categorical_features.remove("brand_name")
            print("   - brand_name은 벡터화되어 범주형에서 제외")
        
        for col in categorical_features + numeric_features:
            if col in self.train.columns:
                self.train_vectorized[col] = self.train[col].reset_index(drop=True)
                self.test_vectorized[col] = self.test[col].reset_index(drop=True)
        
        gc.collect()
        
        print(f"✅ 벡터화 완료!")
        print(f"   - train: {self.train_vectorized.shape}")
        print(f"   - test: {self.test_vectorized.shape}")
        print(f"   - 최종 피처 수: {len(feature_names) + len(categorical_features) + len(numeric_features)}")

    def setup_pycaret(self, session_id=23, fold=3, use_gpu=False):
        """
        PyCaret 환경 설정 (32GB RAM 최적화)
        
        fold=3: CV 폴드 수 축소로 메모리 절약
        use_gpu=False: CPU 전용 (Ryzen 5 Pro 4650)
        """
        print("🔧 PyCaret setup 시작...")
        
        categorical_cols = [
            "main_cat", "sub_cat", "sub_sub_cat", 
            "brand_name", "item_condition_id", "shipping"
        ]
        existing_categorical = [col for col in categorical_cols 
                               if col in self.train_vectorized.columns]

        self.setup_result = setup(
            data=self.train_vectorized.assign(
                price=self.train["price"].reset_index(drop=True)
            ),
            target="price",
            session_id=session_id,
            categorical_features=existing_categorical if existing_categorical else None,
            normalize=True,
            transformation=False,
            fold_strategy="kfold",
            fold=fold,  # 3-fold CV
            use_gpu=use_gpu,
            n_jobs=4,  # Ryzen 5 Pro 4650 (6코어 중 4개 사용)
            verbose=True,
            html=False
        )
        
        gc.collect()
        print("✅ PyCaret setup 완료")

    def find_and_blend_models(self, top_n=3, sort_metric="R2"):
        """
        상위권 모델 탐색 및 블렌딩
        
        top_n=3: 메모리 절약을 위해 상위 3개만 선택
        """
        if self.setup_result is None:
            raise ValueError("먼저 setup_pycaret()를 실행하세요.")

        print(f"🔍 상위 {top_n}개 모델 탐색 시작...")
        
        # 1단계: 빠른 모델 비교
        top_models = compare_models(
            n_select=top_n,
            sort=sort_metric,
            turbo=True,  # 빠른 학습 모드
            verbose=True
        )
        
        print(f"✅ 상위 {top_n}개 모델 선정 완료")
        
        # 개별 모델이 리스트로 반환되는지 확인
        if not isinstance(top_models, list):
            top_models = [top_models]
        
        print("🎯 선정된 모델:")
        for i, model in enumerate(top_models, 1):
            print(f"   {i}. {str(model).split('(')[0]}")
        
        # 2단계: 블렌딩
        print(f"\n🔀 {len(top_models)}개 모델 블렌딩 시작...")
        blended = blend_models(
            estimator_list=top_models,
            optimize=sort_metric,
            choose_better=True,
            verbose=True
        )
        
        self.best_model = blended
        gc.collect()
        
        print(f"🏆 Blended model 생성 완료 (기준={sort_metric})")
        return self.best_model

    def save_metrics(self, model_name=None):
        """성능 지표 계산 및 저장 (실제 가격 스케일)"""
        if self.best_model is None:
            raise ValueError("모델이 없습니다.")

        print("📊 성능 평가 중...")
        pred_df = predict_model(self.best_model, data=self.train_vectorized.copy())
        
        y_log_true = self.train["price"].values
        y_log_pred = pred_df["prediction_label"].values

        # 로그 스케일 → 실제 가격 스케일 복원
        y_true = np.expm1(y_log_true)
        y_pred = np.expm1(y_log_pred)

        r2 = r2_score(y_true, y_pred)
        rmse = mean_squared_error(y_true, y_pred, squared=False)
        mae = mean_absolute_error(y_true, y_pred)

        self.metrics = {
            "R2": round(r2, 4), 
            "RMSE": round(rmse, 4), 
            "MAE": round(mae, 4)
        }

        timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
        if model_name is None:
            model_name = str(self.best_model).split("(")[0]

        file_path = os.path.join(self.results_dir, f"{model_name}_metrics_{timestamp}.json")
        with open(file_path, "w") as f:
            json.dump(self.metrics, f, indent=4)

        print(f"💾 Metrics 저장 완료: {file_path}")
        print(f"   - R² = {self.metrics['R2']}")
        print(f"   - RMSE = ${self.metrics['RMSE']:.2f}")
        print(f"   - MAE = ${self.metrics['MAE']:.2f}")

    def visualize_model(self, plots=["residuals", "feature"]):
        """모델 시각화"""
        if self.best_model is None:
            raise ValueError("먼저 find_and_blend_models()로 모델을 선택하세요.")

        print("🎨 시각화 시작...")
        timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
        model_name = str(self.best_model).split("(")[0]

        for p in plots:
            try:
                plot_name = "feature" if p == "feature_importance" else p
                save_path = os.path.join(
                    self.images_dir, 
                    f"{model_name}_{plot_name}_{timestamp}.png"
                )
                plot_model(self.best_model, plot=plot_name, save=True)
                print(f"✅ {plot_name} plot 저장 완료")
            except Exception as e:
                print(f"⚠️ Plot {p} 실패: {e}")

    def predict_test(self, submission_file="submission.csv"):
        """테스트 데이터 예측 및 제출 파일 생성"""
        if self.best_model is None:
            raise ValueError("먼저 find_and_blend_models()로 모델을 선택하세요.")

        print("📦 Test 데이터 예측 시작...")
        predictions = predict_model(self.best_model, data=self.test_vectorized.copy())

        # 로그 스케일 → 실제 가격 복원
        price_log_pred = predictions["prediction_label"].values
        price_pred = np.expm1(price_log_pred)

        submission = pd.DataFrame({
            "test_id": self.test["test_id"], 
            "price": price_pred
        })

        timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
        submission_path = os.path.join(
            self.results_dir, 
            f"{timestamp}_{submission_file}"
        )
        submission.to_csv(submission_path, index=False)
        
        print(f"💾 Submission 저장 완료: {submission_path}")
        print(f"   - 예측 가격 범위: ${price_pred.min():.2f} ~ ${price_pred.max():.2f}")
        print(f"   - 평균 가격: ${price_pred.mean():.2f}")
        
        return submission

# end of class ###############################################################################################

In [11]:
print("=" * 60)
print("Mercari Price Suggestion - HashingVectorizer 버전")
print("32GB RAM 환경 최적화")
print("=" * 60)

# 1. 분석기 초기화
analyzer = MercariPyCaretAnalyzer(
  data_dir="../data",
  images_dir="../images",
  results_dir="../results"
)


Mercari Price Suggestion - HashingVectorizer 버전
32GB RAM 환경 최적화


In [ ]:
# 2. 데이터 로딩 (25% 샘플링 - 32GB RAM 최적)
analyzer.load_data(
  train_file="train.tsv",
  test_file="test.tsv",
  undersample_frac=0.25
)

📂 데이터 로딩 시작...
⚠️ Stratified undersampling 적용: train (370415, 8)
🔄 희귀값 통합 중...
✅ 데이터 로드 완료: train (370415, 14), test (693359, 13)


In [13]:
# 3. HashingVectorizer 기반 텍스트 벡터화
analyzer.vectorize_text_hashing(
  text_columns=["name", "item_description"],
  n_features=2**18,  # 262K features
  n_components=80    # SVD 압축
)

🔍 HashingVectorizer 기반 텍스트 벡터화 시작...
   - n_features: 262,144 (메모리 고정)
   - SVD 압축: 80차원


텍스트 컬럼 처리:   0%|          | 0/2 [00:00<?, ?it/s]

   - name: sparse matrix shape = (370415, 262144)
   - name: SVD 설명력 = 24.86%


텍스트 컬럼 처리:  50%|█████     | 1/2 [00:39<00:39, 39.26s/it]

   - item_description: sparse matrix shape = (370415, 262144)
   - item_description: SVD 설명력 = 37.32%


텍스트 컬럼 처리: 100%|██████████| 2/2 [02:08<00:00, 64.46s/it]


🔗 피처 결합 중...
✅ 벡터화 완료!
   - train: (370415, 170)
   - test: (693359, 170)
   - 최종 피처 수: 170


In [14]:
# 4. PyCaret 환경 설정
analyzer.setup_pycaret(
  session_id=23,
  fold=3,
  use_gpu=False
)


🔧 PyCaret setup 시작...
                    Description             Value
0                    Session id                23
1                        Target             price
2                   Target type        Regression
3           Original data shape     (370415, 171)
4        Transformed data shape     (370415, 185)
5   Transformed train set shape     (259290, 185)
6    Transformed test set shape     (111125, 185)
7              Numeric features               164
8          Categorical features                 6
9                    Preprocess              True
10              Imputation type            simple
11           Numeric imputation              mean
12       Categorical imputation              mode
13     Maximum one-hot encoding                25
14              Encoding method              None
15                    Normalize              True
16             Normalize method            zscore
17               Fold Generator             KFold
18                  Fold Num

In [ ]:

# 5. 상위 3개 모델 탐색 및 블렌딩
analyzer.find_and_blend_models(
  top_n=3,
  sort_metric="R2"
)


🔍 상위 3개 모델 탐색 시작...


Processing:  68%|██████▊   | 59/87 [56:44<1:34:08, 201.72s/it]

In [ ]:
# 6. 성능 평가 및 저장
analyzer.save_metrics()

# 7. 시각화
analyzer.visualize_model(plots=["residuals", "feature"])

# 8. 테스트 예측 및 제출 파일 생성
analyzer.predict_test(submission_file="submission.csv")

print("\n✅ 전체 파이프라인 완료!")

### 중간단계 진행중 의문점

```python
# 5. 상위 3개 모델 탐색 및 블렌딩
analyzer.find_and_blend_models(  top_n=3,  sort_metric="R2")
```
에서 cpu 6개를 100% 다 쓰고 있고 20분째  아래에서 더 진행이 안되고 있어
대신 메모리 사용량은 오르락 내리락 하고 있기는 해... 잘 진행되고 있다고 봐야할까?

🔍 상위 3개 모델 탐색 시작... 
Processing:  59%|█████▊    | 51/87 [09:03<11:49, 19.71s/it]
---
네, **정상적으로 진행되고 있습니다!** 👍 걱정하지 마세요. 상황을 자세히 분석해드릴게요.

## ✅ 정상 동작 신호들

1. **CPU 100% 사용** ✓
   - `n_jobs=4` 설정으로 4개 코어를 풀가동 중
   - 나머지 2개 코어는 시스템/백그라운드 작업
   - **이게 정상입니다!** ML 학습은 CPU intensive

2. **메모리 오르락내리락** ✓
   - 각 모델 학습 → 메모리 증가
   - 모델 완료 → 가비지 컬렉션 → 메모리 감소
   - 3-fold CV 반복 → 메모리 패턴 반복

3. **Progress: 59% (51/87)** ✓
   - 총 87개 작업 = 29개 모델 × 3-fold CV
   - 현재 51개 완료 = 약 17개 모델 완료
   - **약 60% 진행 중**

## 📊 예상 시간 계산

```python
# 현재 상태
경과 시간: 20분 (09:03)
진행률: 59% (51/87)
남은 작업: 36개

# 예상 완료 시간
평균 속도: 20분 / 51작업 = 23.5초/작업
남은 시간: 36 × 23.5초 = 14.1분

총 예상 시간: 20분 + 14분 = 약 34분
```

**하지만 프로그레스 바는 "11:49 남음"이라고 표시 중** → 이건 PyCaret이 보수적으로 추정한 것

## 🔍 현재 무슨 일이 일어나고 있나?

```python
compare_models(n_select=3, turbo=True)

# 내부 동작
1. 29개 회귀 모델 후보 생성
   - Linear Regression, Ridge, Lasso
   - ElasticNet, LARS, Huber
   - KNN, Decision Tree
   - Random Forest, Extra Trees
   - AdaBoost, Gradient Boosting
   - XGBoost, LightGBM, CatBoost
   - SVM, MLP, ...

2. 각 모델마다 3-fold CV 수행
   - fold 1 학습 → 검증
   - fold 2 학습 → 검증
   - fold 3 학습 → 검증
   - 평균 성능 계산

3. R2 기준으로 정렬 → 상위 3개 선택
```

## ⏰ 시간이 오래 걸리는 이유

| 단계 | 소요 시간 | 설명 |
|------|----------|------|
| 빠른 모델 (Linear, Ridge) | 10-30초 | 단순 모델 |
| 중간 모델 (Random Forest) | 1-3분 | 트리 기반 |
| **느린 모델 (XGBoost, CatBoost, LightGBM)** | **3-8분** | **부스팅 알고리즘** |
| SVM, MLP | 2-5분 | 최적화 반복 |

**51/87 진행 중이면 지금 느린 모델들(부스팅 계열)을 학습하고 있을 가능성이 높습니다.**

## 🎯 확인 방법

터미널/콘솔에서 이런 메시지가 보이나요?

```
Fitting 3 folds for each of 29 candidates, totalling 87 fits
[Parallel(n_jobs=4)]: Using backend LokyBackend with 4 concurrent workers.
[Parallel(n_jobs=4)]: Done  42 tasks      | elapsed:  8.5min
[Parallel(n_jobs=4)]: Done  51 tasks      | elapsed: 10.2min
```

보인다면 **정상입니다!**

## ⚡ 다음에 더 빠르게 하려면

```python
# 옵션 1: 더 적은 모델만 비교
analyzer.find_and_blend_models(
    top_n=3,
    sort_metric="R2",
    include=["lightgbm", "xgboost", "catboost", "ridge", "et"]  # 5개만
)

# 옵션 2: turbo=True로 빠른 학습
# (이미 적용되어 있음)

# 옵션 3: fold 수 줄이기
analyzer.setup_pycaret(fold=2)  # 3 → 2로
```

## 📌 결론

**지금 상태: 정상 진행 중** ✅
- CPU 100% → 정상
- 메모리 변동 → 정상
- 59% 진행 → 정상
- **예상 남은 시간: 10-15분**

커피 한 잔 하시고 기다리시면 됩니다! ☕

혹시 30분 후에도 59%에서 멈춰있다면 그때 다시 말씀해주세요. 그럴 가능성은 거의 없지만, 혹시 모르니까요! 😊